# Teil 2 - Datenbeschreibung und Vorbereitung

Notebook: `data_description.ipynb`  
Datensatz: `Steel_industry_data.csv` (Trennzeichen `;`)


## Setup & Daten laden (ohne pandas)
Wir lesen die CSV-Datei mit dem Standardmodul `csv` und bereiten die Werte vor.


In [ ]:
import csv
import math
import random
from collections import Counter
from datetime import datetime
from statistics import mean, median, stdev
import matplotlib.pyplot as plt

DATA_PATH = "Steel_industry_data.csv"

rows = []
with open(DATA_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for row in reader:
        rows.append(row)

len(rows)


## Teilaufgabe 2.1
**Zielvariable:** Ich möchte den Energieverbrauch vorhersagen. Als Zielvariable
eignet sich deshalb **`Usage_kWh`**, da diese Spalte den Verbrauch pro 15-Minuten-Intervall
direkt abbildet. Die übrigen Felder (Blindleistung, Leistungsfaktoren, NSM,
Wochentag, Lasttyp usw.) können als Eingangsmerkmale verwendet werden.


## Teilaufgabe 2.2
Für jedes Feld bestimme ich relevante statistische Kennzahlen. Bei numerischen
Spalten berechne ich Median, Standardabweichung, Minimum/Maximum und den Mittelwert.
Bei kategorialen Spalten bestimme ich die Häufigkeiten.


In [ ]:
# Numerische Spalten vorbereiten
numeric_cols = [
    "Usage_kWh",
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "CO2(tCO2)",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM"
]

def to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

numeric_data = {col: [] for col in numeric_cols}
for row in rows:
    for col in numeric_cols:
        val = to_float(row.get(col, ""))
        if val is not None:
            numeric_data[col].append(val)

# Statistik berechnen
def safe_stdev(values):
    return stdev(values) if len(values) > 1 else 0.0

for col in numeric_cols:
    vals = numeric_data[col]
    print(
        f"{col}: count={len(vals)} | mean={mean(vals):.3f} | median={median(vals):.3f} | std={safe_stdev(vals):.3f} | min={min(vals):.3f} | max={max(vals):.3f}"
    )


In [ ]:
# Fehlende Werte pro Spalte
missing_counts = {}
for key in rows[0].keys():
    missing_counts[key] = sum(1 for r in rows if (r.get(key, "") is None or r.get(key, "").strip() == ""))
missing_counts


In [ ]:
# Kategoriale Spalten (Häufigkeiten)
categorical_cols = ["WeekStatus", "Day_of_week", "Load_Type"]
for col in categorical_cols:
    counts = Counter(r[col] for r in rows)
    print(f"\n{col}")
    for k, v in counts.items():
        print(f"  {k}: {v}")


### Ergebnisse (aus diesem Datensatz, n=999)
Numerische Übersicht (gerundet):

| Feld | Mean | Median | Std | Min | Max |
|---|---:|---:|---:|---:|---:|
| Usage_kWh | 34.482 | 4.64 | 46.622 | 3.10 | 147.46 |
| Lagging_Current_Reactive.Power_kVarh | 14.270 | 4.57 | 19.449 | 0.00 | 79.70 |
| Leading_Current_Reactive_Power_kVarh | 4.933 | 0.00 | 8.918 | 0.00 | 27.04 |
| CO2(tCO2) | 0.013 | 0.00 | 0.022 | 0.00 | 0.07 |
| Lagging_Current_Power_Factor | 84.269 | 91.12 | 14.324 | 53.26 | 100.00 |
| Leading_Current_Power_Factor | 82.022 | 100.00 | 32.464 | 15.82 | 100.00 |
| NSM | 41783.784 | 41400 | 25006.827 | 0.00 | 85500.00 |

Kategoriale Verteilung:

- WeekStatus: Weekday = 807, Weekend = 192
- Day_of_week: Monday = 192, Tuesday = 192, Wednesday = 192, Thursday = 135, Friday = 96, Saturday = 96, Sunday = 96
- Load_Type: Light_Load = 604, Medium_Load = 227, Maximum_Load = 168

**Kurzfazit:** `Usage_kWh` hat eine deutlich höhere Streuung als die Leistungsfaktoren.
Median und Mittelwert liegen weit auseinander, was auf eine starke Rechtsschiefe
(viele kleine Werte, wenige sehr große) hindeutet.


## Teilaufgabe 2.3
Ich erstelle mindestens eine Grafik. Hier ein Histogramm für `Usage_kWh`
und zusätzlich eine Regressionsgrafik zwischen Blindleistung und Verbrauch.


In [ ]:
# Histogramm
plt.figure(figsize=(8, 4))
plt.hist(numeric_data["Usage_kWh"], bins=40, color="#2a6f97", alpha=0.85)
plt.title("Verteilung von Usage_kWh")
plt.xlabel("Usage_kWh")
plt.ylabel("Anzahl")
plt.tight_layout()
plt.show()


In [ ]:
# Regressionsgrafik (einfache lineare Regression ohne numpy/pandas)
x_vals = numeric_data["Lagging_Current_Reactive.Power_kVarh"]
y_vals = numeric_data["Usage_kWh"]

# Stichprobe, damit das Plotten schneller ist
pairs = list(zip(x_vals, y_vals))
sample = random.sample(pairs, k=min(2000, len(pairs)))
xs, ys = zip(*sample)

x_mean = mean(xs)
y_mean = mean(ys)
num = sum((x - x_mean) * (y - y_mean) for x, y in sample)
den = sum((x - x_mean) ** 2 for x in xs)
slope = num / den if den != 0 else 0.0
intercept = y_mean - slope * x_mean

x_line = [min(xs), max(xs)]
y_line = [intercept + slope * x for x in x_line]

plt.figure(figsize=(6, 4))
plt.scatter(xs, ys, s=20, alpha=0.35)
plt.plot(x_line, y_line, color="#d62828")
plt.title("Zusammenhang Blindleistung vs. Verbrauch")
plt.xlabel("Lagging_Current_Reactive.Power_kVarh")
plt.ylabel("Usage_kWh")
plt.tight_layout()
plt.show()


## Teilaufgabe 2.4
Ich skaliere mindestens ein Datenfeld. Ich verwende eine MinMax-Skalierung für `NSM`,
da dieser Wert einen deutlich größeren Wertebereich hat als andere Merkmale und
sonst zu stark gewichtet wäre.


In [ ]:
nsm_vals = numeric_data["NSM"]
nsm_min = min(nsm_vals)
nsm_max = max(nsm_vals)
nsm_scaled = [(v - nsm_min) / (nsm_max - nsm_min) for v in nsm_vals]
nsm_scaled[:5]
